In [1]:
import copy
import functools
import glob
import pickle
import scipy
import sklearn
import warnings

warnings.filterwarnings('ignore')

import graphviz as gr
import numpy as np
import os
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm

from matplotlib import style
from matplotlib import pyplot as plt
from pprint import pprint
from scipy import stats, special
from sklearn import mixture
from sklearn import datasets
from sklearn.linear_model import LinearRegression
from statsmodels.regression.rolling import RollingOLS

from tqdm import tqdm

%matplotlib inline

style.use("fivethirtyeight")
pd.set_option("display.max_columns", 6)

np.random.seed(0)

In [2]:
#!pip freeze > ../requirements.txt
#!pip install -r ../requirements.txt --user

# Pre-Process Colorado Data: Attach fips to them

In [3]:
DATA_FOLDER = "../data"
COLORADO_DATA_FOLDER = os.path.join(DATA_FOLDER,"Colorado_Data")
COLORADO_DF_PATH = os.path.join(COLORADO_DATA_FOLDER,"Colorado_Outbreak_Data_2023-04-26.csv")
MASTER_FIPS_PATH = os.path.join(COLORADO_DATA_FOLDER,"county_fips_master.csv")

### Clean up the Colorado Dataset

In [4]:
colorado_df_orig = pd.read_csv(COLORADO_DF_PATH,parse_dates=True)

# Shortern Column Names
rename_cols = {"Colorado county (exposure location)":"County", "Date reported to public health":"Date Reported", "Date outbreak was considered closed:": "Date Resolved"}
rename_cols["Is this an At Risk Population? (Healthcare, Corrections, Highly Mobile, Schools, Critical Infrastructure)"]="At Risk"
colorado_df = colorado_df_orig.rename(columns=rename_cols)
# Drop Unnamed Columns and others
colorado_df = colorado_df.drop(colorado_df.filter(regex='Unnamed').columns, axis=1)
#colorado_df = colorado_df.drop(columns=["Investigation status"],axis=1)
colorado_df = colorado_df.drop(columns=["COVID Setting Type"],axis=1)
colorado_df = colorado_df.drop(columns=["If setting type is other, specify"],axis=1)
#colorado_df = colorado_df.drop(columns=["Date Resolved"],axis=1)

# Only keep Total cases and Total deaths
drop_total_col = colorado_df.filter(regex='Total').columns
drop_total_col = drop_total_col[:-2]
colorado_df = colorado_df.drop(drop_total_col, axis=1)
# Impute NaN with 0
colorado_df["Date Resolved"] = colorado_df["Date Resolved"].dropna()
colorado_df["Date Reported"] = colorado_df["Date Reported"].dropna()                                          
colorado_df["Date Resolved"] = colorado_df[colorado_df["Date Resolved"] != 0]["Date Resolved"]
colorado_df["Date Reported"] = colorado_df[colorado_df["Date Reported"] != 0]["Date Reported"]
colorado_df = colorado_df.replace(np.nan,0)
# Change at Risk to Booleans
#colorado_df["At Risk"] = (colorado_df["At Risk"] == "Yes")

colorado_df.head()

,Setting name,Date Resolved,County,Date Reported,Total cases,Total deaths
0,Bridge Tournament,2020-04-14,El Paso,2020-03-14,24,4
1,"Fairacres Manor, Inc. (020369): March 2020",2020-05-18,Weld,2020-03-17,86,13
2,North Shore Health & Rehab Facility (020331): ...,2020-05-19,Larimer,2020-03-17,42,11
3,Brookdale Meridian Englewood (020409): April 2020,2020-07-28,Arapahoe,2020-03-18,35,10
4,The Residence at Oakridge (23R289): April 2020,2020-06-02,Larimer,2020-03-18,15,1


### Prepare the Master Fips file

In [5]:
master_fips = pd.read_csv(MASTER_FIPS_PATH)
# Keep only Colorado
master_fips = master_fips[master_fips["state_abbr"]=="CO"]
master_fips = master_fips[["fips", "county_name"]]
master_fips["county_name"] = master_fips["county_name"].str.split(" ").str[0]
master_fips = master_fips.rename(columns={"county_name":"County"})
master_fips.head()

,fips,County
245,8001,Adams
246,8003,Alamosa
247,8005,Arapahoe
248,8007,Archuleta
249,8009,Baca


In [6]:
master_fips[master_fips["County"]=="Weld"]


,fips,County
307,8123,Weld


### Join `master_fips` with `colorado_df` on `left="county_name", right="county"`

In [7]:
colorado_df_processed = pd.merge(left=colorado_df, right=master_fips, how="inner", on="County")
colorado_df_processed = colorado_df_processed[colorado_df_processed["Date Resolved"] != 0]
colorado_df_processed = colorado_df_processed[colorado_df_processed["Date Reported"] != 0]
colorado_df_processed.to_csv(os.path.join(DATA_FOLDER,"colorado_outbreaks_2023-04-26.csv"),index=False)


# Process the Backtest Data of Colorado (lm and grf results)

In [8]:
#BACKTEST_FOLDER = os.path.join(DATA_FOLDER,"Backtest_by_County_wsize2_vaccination")
#BACKTEST_FOLDER = os.path.join(DATA_FOLDER,"Backtest_by_County_updated_cumulative_cases")
BACKTEST_FOLDER = os.path.join(DATA_FOLDER,"Backtest_by_County_wsize7_atleast_20")
WSIZE_FOLDER = "wsize7_atleast_20_backup"
BACKTEST_FNAMES_ARRAY = os.listdir(BACKTEST_FOLDER)

FileNotFoundError: [Errno 2] No such file or directory: '../data/Backtest_by_County_wsize7_atleast_20'

In [ ]:
onlyfiles

In [ ]:
# Get the backtest fnames matching the Colorado FIPS numbers
#colorado_backtest_fnames_array = master_fips["fips"].astype(str) + "_backtest.csv"
#colorado_backtest_fnames_array = [os.path.join(BACKTEST_FOLDER,fname) for fname in colorado_backtest_fnames_array]
onlyfiles = [os.path.join(BACKTEST_FOLDER, f) for f in os.listdir(BACKTEST_FOLDER) if os.path.isfile(os.path.join(BACKTEST_FOLDER, f))]

colorado_backtest_df_orig = pd.concat(map(pd.read_csv, onlyfiles))

# Drop columns
dropcols = ['D.r.lm', 'D.tau.hat', 'B.D.r.lm','B.D.tau.hat']
#dropcols += ["D.r.slm","B.D.r.slm"]
#dropcols += ['tau.variance','tau.upr', 'tau.lwr']
dropcols += ['predicted.grf.future.0', "state"]
colorado_backtest_df = colorado_backtest_df_orig.drop(columns=dropcols,axis=1)
# Rename tau.hat to r.grf
renamecols = {"tau.hat":"r.grf", "predicted.grf.future.last":"predicted.grf"}
colorado_backtest_df = colorado_backtest_df.rename(columns=renamecols)

# Write file
colorado_backtest_df.to_csv(os.path.join(DATA_FOLDER,"colorado_backtest.csv"), index=False)

colorado_backtest_df


In [ ]:
colorado_backtest_df_orig.columns

# Process Backtest Data for All States

In [ ]:
# Get the backtest fnames for all states
all_backtest_fnames_array = [os.path.join(BACKTEST_FOLDER,fname) for fname in BACKTEST_FNAMES_ARRAY]
all_backtest_df_orig = pd.concat(map(pd.read_csv, all_backtest_fnames_array))
# Drop columns
dropcols = ['D.r.lm', 'D.tau.hat', 'B.D.r.lm','B.D.tau.hat']
dropcols += ['predicted.grf.future.0', "state"]
#dropcols += ["D.r.slm","B.D.r.slm"]
all_backtest_df = all_backtest_df_orig.drop(columns=dropcols,axis=1)
# Rename tau.hat to r.grf
renamecols = {"tau.hat":"r.grf", "predicted.grf.future.last":"predicted.grf"}
all_backtest_df = all_backtest_df.rename(columns=renamecols)

# Rolling OLS per group
all_backtest_df["date.y"] = pd.to_datetime(all_backtest_df['date.y'])  
all_backtest_df['date_delta'] = (all_backtest_df['date.y'] - all_backtest_df['date.y'].min())  / np.timedelta64(1,'D')
all_backtest_df["r.gt.y"] = -999
for fips in tqdm(all_backtest_df["fips"].unique()):
    df_mask = all_backtest_df["fips"]==fips
    df = all_backtest_df[df_mask]
    y = df[['log_rolled_cases.y']].values
    X = df[['date_delta']].values
    X = sm.add_constant(X)
    
    if np.shape(y)[0] < 7:
        continue
    model = RollingOLS(endog=y, exog=X, window=7)
    results = model.fit(params_only=True)
    df['r.gt.y'] = (results.params[:,1]).flatten()
    all_backtest_df.loc[df_mask,"r.gt.y"] = df['r.gt.y']
# Write
all_backtest_df = all_backtest_df[all_backtest_df["r.gt.y"] != -999]
all_backtest_df.to_csv(os.path.join(DATA_FOLDER,WSIZE_FOLDER,"all_backtest.csv"), index=False)

In [ ]:
all_backtest_df.columns

In [ ]:
all_backtest_df

In [ ]:
fips

In [ ]:
all_backtest_df[all_backtest_df["fips"]==fips]

In [ ]:
np.shape(y)